# Importing Important modules

In [ ]:
from obspy import read_inventory
from obspy import read_events

import seisbench.data as sbd

from datetime import datetime
from pathlib import Path
import logging
import os

In [ ]:
import sys
lib_path = [
    r'C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisRoutine',
    r'C:\Users\ikahb\OneDrive\Applications\GitHub\SeisRoutine',
]
for path in lib_path:
    sys.path.append(path)
##########################################################################
import SeisRoutine.catalog as src
import SeisRoutine.waveform as srw
import SeisRoutine.config as srconf
import SeisRoutine.seisbench as srsb

In [ ]:
import warnings
warnings.simplefilter('ignore', DeprecationWarning)

In [ ]:
# from importlib import reload  # Python 3.4+
# src = reload(src)
# srw = reload(srw)
# srconf = reload(srconf)
# srsb = reload(srsb)

# Initializing the init file and starting logging.

In [ ]:
init_cfg = srconf.Config.load('0-init-cfg.yml')
cfg_path = os.path.join(
    init_cfg.target_config_filepath,
    init_cfg.target_config_filename
)

timestamp = srconf.timestamp()
cfg = srconf.Config.load(cfg_path, resolve=True)
cfg.resolve(context={"timestamp": timestamp})

with open(cfg_path.replace('cfg', 'cfg_LastRun'), 'w') as file:
    cfg.to_yaml(stream=file, default_flow_style=False, indent=4)

In [ ]:
srconf.configure_logging(**cfg.to_dict()['log'])

In [ ]:
running_file_info = srconf.RuntimeLocation.get_caller_info()
msg = f"Logging has started for notebook:\n{running_file_info['full_path']}"
logging.info(msg)

# List all installed packages and their versions
msg = srconf.EnvironmentInfo().report(include_freeze=True)
logging.info(msg)

msg = cfg.__str__()
logging.info(f'Configuration File:\n{msg}')

# Loading Seismic Catalog and network details.

In [ ]:
catalog = read_events(cfg.dataset.path.catalog)
catalog = [ev for ev in catalog if ev.picks != []]

inventory = read_inventory(cfg.dataset.path.inventory)

stream_cache = srw.waveform.StreamCache(
    root=cfg.dataset.path.stream_root,
    pattern_path=cfg.dataset.path.stream_pattern,
)

In [ ]:
# otime = [ev.preferred_origin().time.timestamp for ev in catalog]
# import matplotlib.pyplot as plt
# _ = plt.hist(otime)

In [ ]:
src.print_phase_frequency(catalog, case_sensitivity=False)

In [ ]:
aca = src.ArrivalCoverageAnalyzer(catalog)
msg = aca.build_message()
logging.info(msg)

#### Writing to SeisBench format

Now, we can combine all the above functions together to write a dataset in SeisBench format. For this, we first need to define the path. For this example, we are using the current working directory. A dataset consists of 2 components:
 - a metadata file, always called `metadata.csv`, which contains all the associated properties of the waveform examples (e.g. trace parameters, source parameters etc.).
 - a waveforms file, always called `waveforms.hdf5`, containing the raw waveforms.

To write the dataset, we use the `WaveformDataWriter` provided by SeisBench. The writer should always be used as a context manager, i.e., using the `with` statement, as shown below. This is to ensure files are properly clsoed after writing and teardown and cleanup operations are always called when exiting the context manager.

First, we need to set the data format for our dataset. We do this by assigning a dictionary to the `writer.data_format` group.

Next, we iterate over all event and all picks in the events. Using the functions above, we generate the event and trace metadata and download the waveforms. We then convert the waveforms to a numpy array using the function `stream_to_array` provided in `seisbench.util`.

As a last step, we hand the event metadata and the waveforms as numpy array over to the writer using `add_trace`. The writer then automatically takes care of writing out the data in the correct format. It also takes care of performance optimisations that we outline in the further considerations below.

In [ ]:
base_path = Path(cfg.dataset.path.dataset)
metadata_path = base_path / "metadata.csv"
waveforms_path = base_path / "waveforms.hdf5"
###
msg = (
    "Dataset will be save at:\n"
    + str(metadata_path)
    + '\n'
    + str(waveforms_path)
)

if cfg.dataset.save_streams.enabled:
    stream_path = base_path / cfg.dataset.save_streams.format
    os.makedirs(stream_path, exist_ok=True)
    msg += '\n'+ str(stream_path)

logging.info(msg)

In [ ]:
# Iterate over events and picks, write to SeisBench format
n_passed_picks = 0
n_all_events = len(catalog)
n_all_picks = aca.stats['with_arrival']
n_events_step = 100
with sbd.WaveformDataWriter(metadata_path, waveforms_path) as writer:
    writer.data_format = cfg.to_dict()['dataset']['data_format']
    for n_passed_events, event in enumerate(catalog, start=1):
        origin = event.preferred_origin()
        _stations = {
            pick.waveform_id.station_code
            for pick in event.picks
        }
        selector = src.CatalogPickArrivalSelector(
            picks=event.picks,
            arrivals=origin.arrivals,
        )
        for station_name in _stations:
            picks = selector.get_picks_by_station(
                station_name=station_name,
                exclude_amplitude=True,
                time_sort=True
            )

            stream = stream_cache.get(
                time=origin.time,
                station_code=station_name
            )

            if picks == [] or len(stream) == 0:
                continue

            ###
            pick = picks[0]
            stime = pick.time - cfg.dataset.trim.before
            etime = pick.time + cfg.dataset.trim.after
            st = stream.slice(
                starttime=stime,
                endtime=etime,
                nearest_sample=True
            )

            # It's possible that all data were masked! If not split,
            # N empty traces exist and len(st) shows N.
            st = st.split()
            ### Check remaining data
            if len(st) == 0:
                msg = (
                    "No waveforms after slicing | "
                    f"Station: {station_name} | "
                    f"Pick: {pick.time} | "
                    f"Origin time: {origin.time} |"
                )
                logging.warning(msg)
                continue

            metadata_builder = srsb.dataset.MetadataBuilder(
                stream=st,
                event=event,
                inventory=inventory,
                component_order=cfg.dataset.data_format.component_order,
                trace_category="earthquake",
            )
            
            metadata = metadata_builder.build_metadata()

            writer.add_trace(
                waveform=metadata_builder.data,
                metadata=metadata,
            )
            ### Saving stream
            if cfg.dataset.save_streams.enabled:
                _format = cfg.dataset.save_streams.format
                otime = origin.time.strftime("%Y-%m-%dT%H-%M-%S")
                filename = f'{n_passed_events-1}_{otime}_{station_name}.{_format}'
                st.write(
                    filename=stream_path/filename,
                    format=_format
                )
            n_passed_picks += len(picks)
        ### Write log
        if n_passed_events % n_events_step == 0:
            msg = srconf.ProgressMsg.build(
                Events=[n_passed_events, n_all_events],
                Picks=[n_passed_picks, n_all_picks]
            )
            logging.info(msg)

msg = srconf.ProgressMsg.build(
    Events=[n_passed_events, n_all_events],
    Picks=[n_passed_picks, n_all_picks]
)
logging.info(msg)